# Lambert Liu Runner

In [3]:
import utils as ut
import b_run_staging as b
import h_ll_runner as h
from i_hyper_tuning import Tuner
import c_clustering as c
import numpy as np
import polars as pl
from numba import set_num_threads, get_num_threads

set_num_threads(15)
get_num_threads()

15

### Loading in required data and changing to named tuples

In [4]:
# Loading in required numpy arrays
static_configs = ut.load_json5('static_configs')
runtime_configs = ut.load_json5('runtime_configs')
base_config = ut.merge_configs(static_configs, runtime_configs)

train_test_dict = ut.load_json5("train_test_dict")
bin_metric_dict = ut.load_json5("bin_metric_dict")

user_counts = ut.load_data("user_counts", "df")
user_interactions = ut.load_data("user_interactions", "df")
user_mapping = ut.load_data("user_mapping", "df")

degen_mask = ut.load_data("degen_mask", "np")
interpolation_weights = ut.load_data("interpolation_weights", "np")

# Loading initial grids
u_init = ut.load_data("u_init", "np")
v_init = ut.load_data("v_init", "np")

p_init = ut.load_data("p_init", "np")
u_pos_init = ut.load_data("u_pos_init", "np")
v_pos_init = ut.load_data("v_pos_init", "np")

n_counts_init = ut.load_data("n_counts_init", "np")

u_clustering = ut.load_data("u_clustering", "np")
v_clustering = ut.load_data("v_clustering", "np")

u_pos_clustering = ut.load_data("u_pos_clustering", "np")
v_pos_clustering = ut.load_data("v_pos_clustering", "np")
p_pos_clustering = ut.load_data("p_pos_clustering", "np")


### Converting to named tuples

In [5]:
# Converting the dfs to nt of numpy arrays to be used for the final numba runner
user_interactions_nt = b.df_to_nt('user_interactions_nt', user_interactions)
user_counts_nt = b.df_to_nt('user_counts_nt', user_counts,)
output_idx_nt, model_idx_nt = (b.get_model_and_output_idx_nt())
train_test_nt_class = b.dictionary_to_named_tuple_class('train_test_nt',train_test_dict)
train_test_nt = train_test_nt_class(**train_test_dict)
bin_metric_nt = b.dictionary_to_named_tuple_class('bin_metric_nt', bin_metric_dict)(**bin_metric_dict)

### Creating tuner class for runs

In [6]:
t = Tuner(u_init, v_init, p_init, u_pos_init, v_pos_init, u_clustering, v_clustering, u_pos_clustering, v_pos_clustering, p_pos_clustering, 
          n_counts_init, user_counts_nt, user_interactions_nt, interpolation_weights, bin_metric_nt, output_idx_nt, model_idx_nt, train_test_nt_class)

### Testing no smoothing NB vs NB hurdle model

In [ ]:
hyperparams = ut.load_json5('hyper_choices')

experiment_name = 'model_selection'
for hurdle_model in (True, False):
    results, calibration_results = t.tune_models(experiment_name=experiment_name, hurdle_model=hurdle_model,  hyperparams=hyperparams, 
                    train_test_dict=train_test_dict, config_dict=base_config, degen_mask=degen_mask, run_name=experiment_name)

finished_config 1/8 in 64.8s


### Validation full tuning Running experiments

# TODO consider that right now we smooth nb parameters based on positive counts only this is probably not what we want.

In [ ]:
hurdle_nb_model = ut.load_json5('hurdle_nb_model')
hyperparams = ut.load_json5('hyper_choices')

if hurdle_nb_model['hurdle_model'] is None:
    raise ValueError('Should only be run after selecting hurdle or NB model')

In [ ]:
experiment_name = 'global_smoothing'

global_results, global_calibration_results = t.tune_models(experiment_name=experiment_name, hurdle_model=hurdle_nb_model['hurdle_model'], 
    hyperparams=hyperparams, train_test_dict=train_test_dict, config_dict=base_config, degen_mask=degen_mask, run_name=experiment_name)

In [ ]:
experiment_name = 'cluster_smoothing'

cluster_results, cluster_calibration_results = t.tune_models(experiment_name=experiment_name, hurdle_model=hurdle_nb_model['hurdle_model'], 
    hyperparams=hyperparams, train_test_dict=train_test_dict, config_dict=base_config, degen_mask=degen_mask, run_name=experiment_name)

## Test final run
Single config test runner

In [ ]:
experiment_name = 'cluster_smoothing'
hurdle_nb_model = ut.load_json5('hurdle_nb_model')
best_model = ut.load_json5('best_configs')
selected_config = best_model[experiment_name]


if selected_config is None:
    raise ValueError(f'No model stored for {experiment_name} in best_configs')

if hurdle_nb_model['hurdle_model'] is None:
    raise ValueError('Should only be run after selecting hurdle or NB model')

# Create one complete runnable configuration
best_config = ut.merge_configs(base_config, hurdle_nb_model, selected_config)


_, config_nt, _, _, _ = b.converting_dicts_to_nt(best_config, train_test_dict, bin_metric_dict)
_, _, _, u_cluster, v_cluster, p_cluster = t.get_ll_param_grids(best_config)
test_model = c.make_cluster_model(cluster_param=best_config['cluster_param'], runtime_configs=best_config, 
                                  u_init=u_cluster, v_init=v_cluster, p_init=p_cluster)

output_metrics, calibration_outputs, *_ = t.run_pipeline_ll(model=test_model, config_nt=config_nt, train_test_nt=train_test_nt, 
                                                            config_dict=best_config, degen_mask=degen_mask)

test_results = [t.make_output_table_row(model=test_model, output_metrics=output_metrics, config_dict=best_config, test_valid='test', 
                                        experiment_name=experiment_name)]

test_calibration_results = t.make_calibration_output_rows(model=test_model, output_metrics=output_metrics, calibration_outputs=calibration_outputs, 
                                                          test_valid='test', config_dict=best_config, experiment_name=experiment_name)

In [ ]:
# Storing results
ut.store_run_results(results=test_results, calibration_results=test_calibration_results, dir='test', run_name=f'{experiment_name}_test')